In [1]:
import os, re, gc
import glob
import pyarrow.parquet as pq
import pandas as pd
import json
import numpy as np

def infer_platform_from_filename(path: str) -> str:
    # details_kr_part1.parquet / timeline_euw1_part2.parquet 같은 패턴 가정
    base = os.path.basename(path).lower()
    m = re.search(r'^(details|timeline)_([a-z0-9]+)_part\d+\.parquet$', base)
    if not m:
        # 혹시 패턴이 다르면 여기서 그냥 'unknown' 두고 파일명 그대로 출력해도 됨
        return "unknown"
    return m.group(2)

def read_parquet(path: str) -> pd.DataFrame:
    return pq.read_table(path).to_pandas()

def safe_write_parquet(df: pd.DataFrame, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_parquet(out_path, index=False)

In [ ]:
# =========================
# 0) helpers
# =========================
def to_py(obj):
    """numpy object/bytes/json-string → python object(list/dict/str/number)"""
    if obj is None or (isinstance(obj, float) and np.isnan(obj)):
        return None
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (bytes, bytearray)):
        obj = obj.decode("utf-8", errors="ignore")
    if isinstance(obj, str):
        s = obj.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return json.loads(s)
            except Exception:
                return obj
        return obj
    return obj

def _get(d, key, default=None):
    return d.get(key, default) if isinstance(d, dict) else default

def _get_obj(obj_dict, obj_name, subkey, default=None):
    """objectives[obj_name][subkey] safe getter"""
    block = obj_dict.get(obj_name) if isinstance(obj_dict, dict) else None
    if not isinstance(block, dict):
        return default
    return block.get(subkey, default)

def _get_stat(p: dict, key: str, default=None):
    """
    participant top-level 우선, 없으면 challenges에서 fallback.
    (예: damageDealtToEpicMonsters / turretPlatesTaken 같은 케이스 대응)
    """
    if not isinstance(p, dict):
        return default
    v = p.get(key, None)
    if v is not None:
        return v
    ch = p.get("challenges", None)
    if isinstance(ch, dict):
        return ch.get(key, default)
    return default

def _get_challenge(p: dict, ch_key: str, default=None):
    """participant['challenges'][ch_key] safe getter"""
    ch = p.get("challenges", None)
    if isinstance(ch, dict):
        return ch.get(ch_key, default)
    return default


# =========================
# 1) schema (1차 확정)
# =========================

# --- match_meta (1 row / match)
META_KEYS = [
    ("queueId", "queue_id"),
    ("gameCreation", "game_creation"),
    ("gameDuration", "game_duration"),
    ("gameMode", "game_mode"),
    ("gameVersion", "game_version"),
    # EDA/정합성 보조(팀 win과 중복이므로 모델 입력 X)
    ("endOfGameResult", "end_of_game_result"),
]

# --- teams (2 rows / match)
TEAM_OBJ_FIELDS = [
    ("atakhan", "obj_atakhan"),
    ("baron", "obj_baron"),
    ("champion", "obj_champion"),  # 필요 없으면 drop 가능
    ("dragon", "obj_dragon"),
    ("horde", "obj_horde"),
    ("inhibitor", "obj_inhibitor"),
    ("riftHerald", "obj_riftHerald"),
    ("tower", "obj_tower"),
]

# --- participants (10 rows / match)
# NOTE: ch_* 는 top-level이 아니라 challenges에서 꺼내야 하므로 별도 목록으로 분리
PART_KEYS_BASE = [
    # identifiers / position
    ("participantId", "participant_id"),
    ("puuid", "puuid"),
    ("teamId", "team_id"),
    ("teamPosition", "team_position"),
    ("individualPosition", "individual_position"),
    ("role", "role"),
    ("championId", "champion_id"),
    ("championName", "champion_name"),
    ("champExperience", "champ_experience"),
    ("champLevel", "champ_level"),
    ("win", "win"),

    # combat core
    ("kills", "kills"),
    ("deaths", "deaths"),
    ("assists", "assists"),
    ("killingSprees", "killing_sprees"),
    ("totalDamageDealt", "total_damage_dealt"),
    ("totalDamageDealtToChampions", "total_damage_dealt_to_champions"),
    ("totalDamageTaken", "total_damage_taken"),
    ("damageSelfMitigated", "damage_self_mitigated"),
    ("totalTimeSpentDead", "total_time_spent_dead"),
    ("totalTimeCCDealt", "total_time_cc_dealt"),
    ("timeCCingOthers", "time_ccing_others"),

    # economy / growth
    ("goldEarned", "gold_earned"),
    ("goldSpent", "gold_spent"),
    ("totalMinionsKilled", "total_minions_killed"),
    ("neutralMinionsKilled", "neutral_minions_killed"),
    ("totalAllyJungleMinionsKilled", "total_ally_jungle_minions_killed"),
    ("totalEnemyJungleMinionsKilled", "total_enemy_jungle_minions_killed"),
    ("timePlayed", "time_played"),

    # objectives contribution (여긴 일부가 challenges에 있을 수도 있어 fallback 적용)
    ("baronKills", "baron_kills"),
    ("dragonKills", "dragon_kills"),
    ("damageDealtToBuildings", "damage_dealt_to_buildings"),
    ("damageDealtToEpicMonsters", "damage_dealt_to_epic_monsters"),
    ("damageDealtToObjectives", "damage_dealt_to_objectives"),
    ("damageDealtToTurrets", "damage_dealt_to_turrets"),
    ("turretTakedowns", "turret_takedowns"),
    ("turretPlatesTaken", "turret_plates_taken"),
    ("inhibitorTakedowns", "inhibitor_takedowns"),
    ("objectivesStolen", "objectives_stolen"),
    ("objectivesStolenAssists", "objectives_stolen_assists"),

    # vision
    ("visionScore", "vision_score"),
    ("wardsPlaced", "wards_placed"),
    ("wardsKilled", "wards_killed"),
    ("detectorWardsPlaced", "detector_wards_placed"),
    ("visionWardsBoughtInGame", "vision_wards_bought_in_game"),

    # EDA/정합성 전용(모델에는 넣지 않음)
    ("gameEndedInEarlySurrender", "game_ended_in_early_surrender"),
    ("gameEndedInSurrender", "game_ended_in_surrender"),
    ("teamEarlySurrendered", "team_early_surrendered"),
]

# challenges 내 키(원래 엑셀에서 ch_* 로 관리하던 것들)
# (challenge_key, dst_col)
CH_KEYS = [
    ("kda", "ch_kda"),
    ("killParticipation", "ch_kill_participation"),
    ("laningPhaseGoldExpAdvantage", "ch_laning_phase_gold_exp_advantage"),
    ("earlyLaningPhaseGoldExpAdvantage", "ch_early_laning_phase_gold_exp_advantage"),
    ("maxCsAdvantageOnLaneOpponent", "ch_max_cs_advantage_on_lane_opponent"),
    ("maxLevelLeadLaneOpponent", "ch_max_level_lead_on_lane_opponent"),
    ("multikills", "ch_multikills"),
    ("soloKills", "ch_solo_kills"),
    ("takedowns", "ch_takedowns"),
    ("takedownsFirstXMinutes", "ch_takedowns_first_x_minutes"),
    ("takedownsAfterGainingLevelAdvantage", "ch_takedowns_after_level_advantage"),
    ("kTurretsDestroyedBeforePlatesFall", "ch_k_turrets_destroyed_before_plates_fall"),
    ("outerTurretExecutesBefore10Minutes", "ch_outer_turret_executes_before_10_minutes"),
    ("quickFirstTurret", "ch_quick_first_turret"),
    ("quickSoloKills", "ch_quick_solo_kills"),
    ("riftHeraldTakedowns", "ch_rift_herald_takedowns"),
    ("scuttleCrabKills", "ch_scuttle_crab_kills"),
    ("jungleCsBefore10Minutes", "ch_jungle_cs_before_10_minutes"),
    ("junglerKillsEarlyJungle", "ch_jungler_kills_early_jungle"),
    ("junglerTakedownsNearDamagedEpicMonster", "ch_jungler_takedowns_near_damaged_epic_monster"),
    ("enemyJungleMonsterKills", "ch_enemy_jungle_monster_kills"),
    ("alliedJungleMonsterKills", "ch_allied_jungle_monster_kills"),
    ("moreEnemyJungleThanOpponent", "ch_more_enemy_jungle_than_opponent"),
    ("highestChampionDamage", "ch_highest_champion_damage"),
    ("highestCrowdControlScore", "ch_highest_crowd_control_score"),
    ("highestWardKills", "ch_highest_ward_kills"),
    ("effectiveHealAndShielding", "ch_effective_heal_and_shielding"),
    ("initialBuffCount", "ch_initial_buff_count"),
    ("initialCrabCount", "ch_initial_crab_count"),
    ("killsNearEnemyTurret", "ch_kills_near_enemy_turret"),
    ("killsUnderOwnTurret", "ch_kills_under_own_turret"),
    ("playedChampSelectPosition", "ch_played_champ_select_position"),

    # EDA/정합성 전용
    ("hadAfkTeammate", "ch_had_afk_teammate"),
]

# 결측을 0으로 채우는 게 자연스러운 "카운트/누적" 컬럼들 (EDA 편의)
ZERO_FILL_COLS = {
    "damage_dealt_to_epic_monsters",
    "turret_plates_taken",
}


# =========================
# 2) main
# =========================
def process_one_detail_parquet(path: str) -> dict[str, pd.DataFrame]:
    platform = infer_platform_from_filename(path)
    df = read_parquet(path)

    if "metadata" not in df.columns or "info" not in df.columns:
        raise ValueError(f"[{path}] 'metadata'/'info' 컬럼이 없음. 실제 컬럼명을 확인 필요.")

    df = df[["metadata", "info"]].copy()
    df["metadata_py"] = df["metadata"].map(to_py)
    df["info_py"] = df["info"].map(to_py)
    df["match_id"] = df["metadata_py"].map(lambda d: d.get("matchId") if isinstance(d, dict) else None)

    meta_rows, part_rows, team_rows = [], [], []

    for _, r in df.iterrows():
        match_id = r["match_id"]
        if match_id is None:
            print(f"경고: match_id가 None인 행이 있습니다. 해당 행은 건너뜁니다. (metadata: {r['metadata']})")
            continue
        info = r["info_py"] if isinstance(r["info_py"], dict) else {}

        # --- match_meta (1 row / match)
        meta_row = {"match_id": match_id, "platform": platform}
        for src, dst in META_KEYS:
            meta_row[dst] = info.get(src)
        meta_rows.append(meta_row)

        # --- participants (10 rows / match)
        participants = to_py(info.get("participants", []))
        if isinstance(participants, list):
            for p in participants:
                if not isinstance(p, dict):
                    continue

                row = {"match_id": match_id, "platform": platform}

                # base stats: top-level, 일부는 challenges fallback 적용
                for src, dst in PART_KEYS_BASE:
                    if src in ("damageDealtToEpicMonsters", "turretPlatesTaken"):
                        row[dst] = _get_stat(p, src, None)
                    else:
                        row[dst] = p.get(src)

                # challenges (ch_*) : challenges dict에서만 읽음
                for ch_key, dst in CH_KEYS:
                    row[dst] = _get_challenge(p, ch_key, None)

                # derived convenience
                tm = p.get("totalMinionsKilled", 0) or 0
                nm = p.get("neutralMinionsKilled", 0) or 0
                row["cs"] = tm + nm

                # join convenience (optional)
                row["queue_id"] = info.get("queueId")
                row["game_duration"] = info.get("gameDuration")
                row["game_version"] = info.get("gameVersion")
                row["game_mode"] = info.get("gameMode")
                row["game_creation"] = info.get("gameCreation")

                # EDA 편의: 특정 카운트류 결측은 0 처리
                for c in ZERO_FILL_COLS:
                    if row.get(c) is None:
                        row[c] = 0

                part_rows.append(row)

        # --- teams (2 rows / match)
        teams = to_py(info.get("teams", []))
        if isinstance(teams, list):
            for t in teams:
                if not isinstance(t, dict):
                    continue

                objectives = t.get("objectives", {})
                if not isinstance(objectives, dict):
                    objectives = {}

                row = {
                    "match_id": match_id,
                    "platform": platform,
                    "team_id": t.get("teamId"),
                    "win": t.get("win"),
                    # join convenience
                    "queue_id": info.get("queueId"),
                    "game_duration": info.get("gameDuration"),
                    "game_version": info.get("gameVersion"),
                    "game_mode": info.get("gameMode"),
                    "game_creation": info.get("gameCreation"),
                }

                for obj_name, prefix in TEAM_OBJ_FIELDS:
                    row[f"{prefix}_first"] = _get_obj(objectives, obj_name, "first", None)
                    row[f"{prefix}_kills"] = _get_obj(objectives, obj_name, "kills", None)

                team_rows.append(row)

    match_meta = pd.DataFrame(meta_rows)
    participants = pd.DataFrame(part_rows)
    teams = pd.DataFrame(team_rows)

    # 권장: team 테이블에서 obj_champion_* 제거(팀킬=participants 합과 중복)
    # drop_cols = ["obj_champion_first", "obj_champion_kills"]
    # teams = teams.drop(columns=[c for c in drop_cols if c in teams.columns], errors="ignore")

    return {
        "match_meta": match_meta,
        "participants": participants,
        "teams": teams,
    }

In [ ]:
def process_one_timeline_parquet(path: str, start_min, end_min) -> dict[str, pd.DataFrame]:
    platform = infer_platform_from_filename(path)
    df = read_parquet(path)

    if "metadata" not in df.columns or "info" not in df.columns:
        raise ValueError(f"[{path}] 'metadata'/'info' 컬럼이 없음. 실제 컬럼명을 확인 필요.")

    df = df[["metadata", "info"]].copy()
    df["metadata_py"] = df["metadata"].map(to_py)
    df["info_py"] = df["info"].map(to_py)
    df["match_id"] = df["metadata_py"].map(lambda d: d.get("matchId") if isinstance(d, dict) else None)

    frames_rows, events_rows = [], []

    for _, r in df.iterrows():
        match_id = r["match_id"]
        if match_id is None:
            continue

        info = r["info_py"] if isinstance(r["info_py"], dict) else {}

        frames = to_py(info.get("frames", []))
        if not isinstance(frames, list):
            continue

        for frame in frames:
            if not isinstance(frame, dict):
                continue

            ts = frame.get("timestamp")
            if not isinstance(ts, (int, float)):
                continue

            minute = int(ts // 60000)
            if minute < start_min or minute > end_min:
                continue

            # -------------------------
            # 1) participantFrames snapshot
            # -------------------------
            pframes = frame.get("participantFrames", {})
            if isinstance(pframes, dict):
                for pid_key, pf in pframes.items():
                    if not isinstance(pf, dict):
                        continue

                    # pid 후보: key -> pf.participantId fallback
                    pid_from_key = None
                    if isinstance(pid_key, (int, float)) and not pd.isna(pid_key):
                        pid_from_key = int(pid_key)
                    else:
                        s = str(pid_key)
                        if s.isdigit():
                            pid_from_key = int(s)

                    pid_from_pf = pf.get("participantId")
                    if isinstance(pid_from_pf, (int, float)) and not pd.isna(pid_from_pf):
                        pid_from_pf = int(pid_from_pf)
                    else:
                        pid_from_pf = None

                    if pid_from_key is not None and 1 <= pid_from_key <= 10:
                        pid = pid_from_key
                    elif pid_from_pf is not None and 1 <= pid_from_pf <= 10:
                        pid = pid_from_pf
                    else:
                        continue

                    dmg = pf.get("damageStats", {})
                    if not isinstance(dmg, dict):
                        dmg = {}

                    frames_rows.append({
                        "match_id": match_id,
                        "platform": platform,
                        "participant_id": pid,
                        "timestamp": ts,
                        "minute": minute,

                        "level": pf.get("level"),
                        "xp": pf.get("xp"),
                        "total_gold": pf.get("totalGold"),
                        "minions_killed": pf.get("minionsKilled"),
                        "jungle_minions_killed": pf.get("jungleMinionsKilled"),
                        "time_enemy_spent_controlled": pf.get("timeEnemySpentControlled"),

                        "dmg_total_done_to_champions": dmg.get("totalDamageDoneToChampions"),
                        "dmg_total_taken": dmg.get("totalDamageTaken"),
                    })

            # -------------------------
            # 2) events
            # -------------------------
            events = to_py(frame.get("events", [])) or []
            if not isinstance(events, list):
                continue

            for e in events:
                if not isinstance(e, dict):
                    continue

                etype = e.get("type")
                ets = e.get("timestamp")
                if not isinstance(ets, (int, float)):
                    continue

                emin = int(ets // 60000)
                if emin < start_min or emin > end_min:
                    continue

                pos = e.get("position", {})
                if not isinstance(pos, dict):
                    pos = {}

                # 공통 스키마(없는건 None)
                row = {
                    "match_id": match_id,
                    "platform": platform,
                    "timestamp": ets,
                    "minute": emin,
                    "event_type": etype,

                    # 공통 participant/team (있으면)
                    "team_id": e.get("teamId"),
                    "killer_id": e.get("killerId"),
                    "creator_id": e.get("creatorId"),
                    "victim_id": e.get("victimId"),

                    # position
                    "position_x": _get(pos, "x"),
                    "position_y": _get(pos, "y"),

                    # combat
                    "assisting_participant_ids": e.get("assistingParticipantIds"),
                    "shutdown_bounty": e.get("shutdownBounty"),
                    "kill_streak_length": e.get("killStreakLength"),

                    # special kill
                    "special_kill_type": e.get("specialKillType"),
                    "multi_kill_length": e.get("multiKillLength"),

                    # objective/building/plate
                    "monster_type": e.get("monsterType"),
                    "monster_sub_type": e.get("monsterSubType"),
                    "kill_type": e.get("killType"),
                    "building_type": e.get("buildingType"),
                    "tower_type": e.get("towerType"),
                    "lane_type": e.get("laneType"),

                    # vision
                    "ward_type": e.get("wardType"),

                    # bounty (있으면)
                    "bounty": e.get("bounty"),
                    "objective": e.get("objective"),
                }

                # 저장하려는 이벤트 타입만 필터링 (원치 않으면 여기 주석 처리)
                KEEP = {
                    "CHAMPION_KILL",
                    "CHAMPION_SPECIAL_KILL",
                    "ELITE_MONSTER_KILL",
                    "BUILDING_KILL",
                    "TURRET_PLATE_DESTROYED",
                    "OBJECTIVE_BOUNTY_PRESTART",
                    "OBJECTIVE_BOUNTY_FINISH",
                    "WARD_PLACED",
                    "WARD_KILL",
                }
                if etype in KEEP:
                    events_rows.append(row)

    return {
        "frames": pd.DataFrame(frames_rows),
        "events": pd.DataFrame(events_rows),
    }

In [ ]:
RAW_DIR = r"../../data/Riot_API/raw/match_detail"
OUT_DIR = r"../../data/Riot_API/processed/match_detail"

paths = sorted(glob.glob(os.path.join(RAW_DIR, "details_me1_part1.parquet")))
# paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.parquet")))

for path in paths:
    base = os.path.basename(path).replace(".parquet", "")
    try:
        out = process_one_detail_parquet(path)

        safe_write_parquet(out["match_meta"], os.path.join(OUT_DIR, "match_meta", f"{base}_meta.parquet"))
        safe_write_parquet(out["participants"], os.path.join(OUT_DIR, "participants", f"{base}_participants.parquet"))
        safe_write_parquet(out["teams"], os.path.join(OUT_DIR, "teams", f"{base}_teams.parquet"))

    finally:
        try:
            del out
        except NameError:
            pass
        gc.collect()

In [ ]:
RAW_DIR = r"../../data/Riot_API/raw/match_timeline"
OUT_DIR = r"../../data/Riot_API/processed/match_timeline"

paths = sorted(glob.glob(os.path.join(RAW_DIR, "timeline_me1_part1.parquet")))
# paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.parquet")))

for path in paths:
    base = os.path.basename(path).replace(".parquet", "")
    try:
        out = process_one_timeline_parquet(path, start_min=0, end_min=100)

        safe_write_parquet(out["frames"], os.path.join(OUT_DIR, "frames", f"{base}_frames.parquet"))
        safe_write_parquet(out["events"], os.path.join(OUT_DIR, "events", f"{base}_events.parquet"))

    finally:
        try:
            del out
        except NameError:
            pass
        gc.collect()